# Assistente virtual em português
DIO · Giordani Martins Silva

Demonstração por texto e arquivo de voz. Execute as células em ordem. Este notebook não tem resultados de voz previamente executados: o serviço de TTS não estava acessível no ambiente de preparação. O kernel Colab não acessa diretamente o microfone do seu computador por PyAudio.

Google Speech recebe o áudio enviado e gTTS recebe o texto. Use gravações próprias sem informação sensível.

In [ ]:
%pip -q install 'SpeechRecognition>=3.14,<4' 'gTTS>=2.5,<3'

In [ ]:
from pathlib import Path
from urllib.request import urlopen
url = 'https://raw.githubusercontent.com/1giordani/trilha-python-dio/main/projetos/assistente-virtual-python/assistente.py'
with urlopen(url, timeout=20) as resposta:
    Path('assistente.py').write_bytes(resposta.read())
from assistente import interpretar, reconhecer, sintetizar
from IPython.display import display, Audio, HTML
from html import escape

## Testar o entendimento e abrir o destino
A célula gera um link. Clique nele para abrir a pesquisa no seu navegador; `webbrowser.open` no servidor remoto não abre uma aba no seu computador.

In [ ]:
comando = 'Pesquisar na Wikipédia sobre inteligência artificial'
acao = interpretar(comando)
print(acao.resposta)
if acao.url:
    display(HTML(f'<a href="{escape(acao.url, quote=True)}" target="_blank" rel="noopener noreferrer">Abrir pesquisa</a>'))

## Texto para áudio
Requer acesso ao serviço Google. O arquivo é MP3. Em caso de falha, a resposta em texto continua disponível.

In [ ]:
try:
    caminho = sintetizar(acao.resposta, 'resposta.mp3')
    display(Audio(filename=str(caminho)))
except Exception as erro:
    print('Áudio indisponível:', erro)

## Fala para texto e comando
Grave uma frase curta (até 15 s) em WAV PCM/AIFF/FLAC. Faça upload somente se concordar em enviar esse áudio ao serviço de reconhecimento. Se o upload não for compatível, converta o arquivo para WAV PCM. A célula abaixo é opcional.

In [ ]:
from google.colab import files
arquivos = files.upload()
if arquivos:
    nome = next(iter(arquivos))
    try:
        transcricao = reconhecer(nome)
        print('Transcrição:', transcricao)
        acao = interpretar(transcricao)
        print(acao.resposta)
        if acao.url:
            display(HTML(f'<a href="{escape(acao.url, quote=True)}" target="_blank" rel="noopener noreferrer">Abrir destino</a>'))
    except Exception as erro:
        print('Não foi possível reconhecer:', erro)

## Limites
PLN por regras, sem treinamento de modelo novo. O mapa apresenta uma busca; não calcula distâncias nem garante a farmácia mais próxima. A execução local com microfone é descrita no README. Os testes de comandos podem ser reproduzidos no computador, com `python -m unittest -v`.